# 05. 실험 2 — α=0 LORO (기준 실험)

**목적 (인수인계 14절):** 학습 8곳에서 한 지역씩 빼고 나머지 7곳으로 학습 → 뺀 지역 평가. 보정 없이(α=0) 지역별 **부호 포함 탄소 오차**, **면적가중 합산 오차**, **평균밀도 기준선**을 잰다. 논문의 "문제 제기" 부분이고, 결과에 따라 주장 범위(전국/시군)가 정해진다(13절).

**이전 05 폐기 이유 (재발 방지 기록)**
- α를 "확률과 균등분포 혼합"으로 잘못 구현 → argmax가 변하지 않아 수학적으로 무효
- 새 6개 지역 캐시를 04와 **다른 피처 정의**로 만들어 기존 3개와 섞음(밴드 스케일 0–1 vs 0–10000, NDMI B11 vs B12) → α=0 결과부터 무효
- 탄소 오차를 부호 없는 tC로 계산, 기준선·합산 없음

**이번 버전**
- 피처·추출·α·탄소오차 함수는 **04_correction.ipynb 원본 그대로** (7-2절 검산 6/6 통과한 정의)
- 22-5절 마스크 추가: 영상 유효(20밴드 합>0) + 결측에서 7px 이내 제외
- 8개 지역 캐시를 같은 조건으로 새로 생성 (`{지역}_s05.npz`, 지역당 60만, 약 70MB). 03/04용 기존 캐시는 건드리지 않음
- 같은 RF로 α 스윕 예측도 저장 → **실험 3에서 RF 재학습 불필요**. 이 노트북은 α=0만 분석

**실행 순서**
1. `모두 실행` → [1]–[5] 진행(캐시 생성은 처음 한 번, 지역당 5–15분), [6]에서 사전 기록 미확정 메시지, [7]에서 멈춤 (정상)
2. [6]의 내용을 확인·수정하고 `CONFIRMED = True` → [6] 실행 (한 번 저장되면 덮어쓰지 않음)
3. [7] 클릭 → `런타임 → 이후 셀 실행`

**끊겼을 때:** `모두 실행` 다시. 캐시는 지역 단위, LORO는 폴드 단위로 저장돼 남은 것만 진행한다. 이 노트북에는 GEE export 셀이 없어 모두 실행해도 안전하다.

In [ ]:
# [1] 설정 — 경로·지역·계수·실험 조건
!pip install -q rasterio

import os, json, time, glob, gc, warnings
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from scipy.ndimage import uniform_filter, distance_transform_edt
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
from google.colab import drive
warnings.filterwarnings('ignore', category=RuntimeWarning)

drive.mount('/content/drive')
D        = '/content/drive/MyDrive/forest/data'
OUT      = '/content/drive/MyDrive/forest/outputs'
FOLD_DIR = f'{OUT}/exp2_folds'
os.makedirs(FOLD_DIR, exist_ok=True)

# 지역 (22-3절 확정). 대전·완도는 시험 지역 → 이 노트북에서 쓰지 않음
TRAIN_REGIONS = ['inje', 'hongcheon', 'gangneung', 'pyeongchang',
                 'bonghwa', 'geumsan', 'geochang', 'suncheon']
LAT   = {'inje': 38.04, 'hongcheon': 37.75, 'gangneung': 37.71, 'pyeongchang': 37.54,
         'bonghwa': 36.92, 'geumsan': 36.13, 'geochang': 35.71, 'suncheon': 35.01}
LARCH = {'inje': 19.0, 'hongcheon': 37.4, 'gangneung': 11.0, 'pyeongchang': 45.8,
         'bonghwa': 16.7, 'geumsan': 19.5, 'geochang': 11.7, 'suncheon': 0.0}

# 탄소계수 [확정] tC/ha. 구계수 {1:55, 2:65, 3:60} 사용 금지
C = {1: 79.25, 2: 101.02, 3: 92.01}

# 실험 조건 (사전 기록 대상)
SEEDS              = [16, 17, 18, 19, 20]
N_MAX              = 600_000   # 지역 캐시 표본 수 (04와 동일)
N_TRAIN_PER_REGION = 30_000    # 학습 지역당 → 7지역 21만 (04의 20만과 비슷한 RF 비용)
ALPHAS             = [0.0, 0.25, 0.40, 0.50, 0.60, 0.75, 1.0]  # 04 [22] 격자. 실험 2는 α=0만 분석
EDGE_PX            = 7         # 22-5: 영상 결측에서 7px 이내 제외

RF_PARAMS = dict(n_estimators=80, min_samples_leaf=20, max_features='sqrt',
                 class_weight='balanced', n_jobs=-1)          # 04와 동일

print('CPU 코어:', os.cpu_count())
print('학습 지역:', TRAIN_REGIONS)
print('탄소계수:', C)

In [ ]:
# [2] 이전 05의 무효 산출물 정리
#   피처 정의 혼재 + 무효 α로 만든 파일이라 재사용 금지. 원본 tif와 03/04용 캐시는 건드리지 않는다.
BAD = ([f'{OUT}/exp2_pool_100k.npz', f'{OUT}/exp2_loro_results.npz', f'{OUT}/exp2_loro_summary.csv']
       + glob.glob(f'{OUT}/exp2_loro_seed*.npz'))
n_del = 0
for p in BAD:
    if os.path.exists(p):
        print(f'  삭제: {os.path.basename(p)} ({os.path.getsize(p)/1e6:.0f} MB)')
        os.remove(p); n_del += 1
print(f'정리 완료 ({n_del}개). 용량을 실제로 돌려받으려면 Drive 휴지통 비우기')

In [ ]:
# [3] 함수 — 04_correction.ipynb 원본 + 22-5 마스크

def make_feats(X):
    # (20,h,w) 반사율(0–1) → (38,h,w). 04 make_feats와 동일 (밴드 순서: B2 B3 B4 B5 B6 B7 B8 B8A B11 B12 ×여름/겨울)
    nd = lambda a, b: (a - b) / (a + b + 1e-6)
    s_ndvi, w_ndvi = nd(X[6], X[2]),  nd(X[16], X[12])   # B8, B4
    s_ndmi, w_ndmi = nd(X[6], X[8]),  nd(X[16], X[18])   # B8, B11
    d_ndvi, d_ndmi = s_ndvi - w_ndvi, s_ndmi - w_ndmi
    f = np.concatenate([X, np.stack([s_ndvi, w_ndvi, s_ndmi, w_ndmi, d_ndvi, d_ndmi])])
    extra = []
    for base in (s_ndvi, w_ndvi, d_ndvi):
        for w in (5, 15):
            a  = np.nan_to_num(base)
            mu = uniform_filter(a, size=w)
            sd = np.sqrt(np.maximum(uniform_filter(a**2, size=w) - mu**2, 0))
            extra += [mu, sd]
    return np.concatenate([f, np.stack(extra)]).astype('float32')


def image_ok_mask(src, H, W, block=2048, edge=EDGE_PX):
    # 22-5 ①: 영상 유효(20밴드 합>0) & 결측에서 edge px 이상 떨어짐
    valid = np.zeros((H, W), bool)
    for r0 in range(0, H, block):
        for c0 in range(0, W, block):
            r1, c1 = min(H, r0 + block), min(W, c0 + block)
            X = src.read(window=Window(c0, r0, c1 - c0, r1 - r0))
            valid[r0:r1, c0:c1] = X.sum(axis=0, dtype='float32') > 0
            del X
    if valid.all():
        return valid
    return distance_transform_edt(valid) >= edge


def extract_samples(name, n_max=N_MAX, block=1024, pad=16, seed=0, TILE=500):
    # 04 extract_samples + 22-5 마스크 + 전체 유효 픽셀 수(n_total) 기록. 캐시: {name}_s05.npz
    cache = f'{OUT}/{name}_s05.npz'
    if os.path.exists(cache):
        z = np.load(cache)
        return {k: z[k] for k in z.files}

    t0 = time.time()
    with rasterio.open(f'{D}/{name}_label.tif') as s: label = s.read(1)
    with rasterio.open(f'{D}/{name}_year.tif')  as s: year  = s.read(1)
    H, W = label.shape
    use0 = (label >= 1) & (label <= 3) & (year >= 2024)
    del year

    with rasterio.open(f'{D}/s2/{name}_s2_20band.tif') as src:
        assert (src.height, src.width) == (H, W), f'{name}: 라벨/영상 격자 불일치'
        scale = 10000.0 if src.dtypes[0] == 'int16' else 1.0      # 22-5 ② (대전만 float)
        use = use0 & image_ok_mask(src, H, W)
        n_excl, n_total = int(use0.sum() - use.sum()), int(use.sum())
        del use0; gc.collect()

        frac  = min(1.0, n_max / max(n_total, 1))
        rng_b = np.random.default_rng(seed)
        Xs, ys, ts = [], [], []
        for r0 in range(0, H, block):
            for c0 in range(0, W, block):
                r1, c1 = min(H, r0 + block), min(W, c0 + block)
                m = use[r0:r1, c0:c1]
                if not m.any(): continue
                sel = m & (rng_b.random(m.shape) < frac)
                if not sel.any(): continue
                pr0, pc0 = max(0, r0 - pad), max(0, c0 - pad)
                pr1, pc1 = min(H, r1 + pad), min(W, c1 + pad)
                X = src.read(window=Window(pc0, pr0, pc1 - pc0, pr1 - pr0)).astype('float32') / scale
                f = make_feats(X)
                f = f[:, r0 - pr0:r0 - pr0 + (r1 - r0), c0 - pc0:c0 - pc0 + (c1 - c0)]
                rr, cc = np.where(sel)
                Xs.append(f[:, sel].T)
                ys.append(label[r0:r1, c0:c1][sel])
                ts.append(((rr + r0) // TILE) * 10000 + (cc + c0) // TILE)
                del X, f

    d = dict(X=np.concatenate(Xs).astype('float32'),
             y=np.concatenate(ys).astype('int8'),
             tile=np.concatenate(ts).astype('int32'),
             n_total=np.int64(n_total), n_excl_edge=np.int64(n_excl),
             scale=np.float32(scale))
    np.savez_compressed(cache, **d)
    print(f'  {name}: 유효 {n_total:,} 중 {len(d["y"]):,}개 저장, 결측경계 제외 {n_excl:,} ({time.time()-t0:.0f}초)')
    return d


def carbon_sum(y):
    return float(sum((y == v).sum() * C[v] for v in (1, 2, 3)))

def carbon_err(y_true, y_pred):
    # 04 carbon_err: 부호 포함 % (예측 − 실제) / 실제
    t = carbon_sum(y_true)
    return (carbon_sum(y_pred) - t) * 100 / t

def stats(X):
    return np.median(X, axis=0), X.std(axis=0) + 1e-8

def shift(Xb, mb, sb, ma, sa, alpha):
    # 7-2절 매개화 [확정]: 평가 지역 b만 학습 통계(a) 쪽으로 α만큼 이동. a = 학습 지역 합친 통계(12절)
    if alpha == 0.0:
        return Xb
    return ((alpha * ma + (1 - alpha) * mb) + (Xb - mb) * (sa / sb) ** alpha).astype('float32')

print('함수 정의 완료')

In [ ]:
# [4] 지역 캐시 생성/로드 — 처음 한 번만 지역당 5–15분. 끊겨도 완료된 지역은 건너뜀
S = {}
for name in TRAIN_REGIONS:
    if not os.path.exists(f'{OUT}/{name}_s05.npz'):
        print(f'{name}: 생성 중...')
    S[name] = extract_samples(name)

rows = []
for name in TRAIN_REGIONS:
    d, y = S[name], S[name]['y']
    rows.append(dict(region=name, lat=LAT[name], larch=LARCH[name],
                     n_total=int(d['n_total']), n_sample=len(y), edge_excl=int(d['n_excl_edge']),
                     conif=(y == 1).mean() * 100, broad=(y == 2).mean() * 100, mixed=(y == 3).mean() * 100,
                     src_scale=float(d['scale'])))
META = pd.DataFrame(rows).set_index('region')
META.to_csv(f'{OUT}/exp2_region_meta.csv')
print(META.round(2).to_string())
print(f'\nRAM 사용 {sum(S[n]["X"].nbytes for n in TRAIN_REGIONS)/1e6:.0f} MB')

In [ ]:
# [5] 검산 — 이전 05의 오류(피처 정의 혼재)가 재발하지 않았는지 확인
cols, labels = [2, 6, 20, 21, 24], ['s_B4', 's_B8', 's_ndvi', 'w_ndvi', 'd_ndvi']
print(f'{"region":12s}' + ''.join(f'{n:>9s}' for n in labels) + '   (숲 s_ndvi 중앙값은 0.87 안팎이어야 함, 22-3절)')
for name in TRAIN_REGIONS:
    med = np.median(S[name]['X'][:, cols], axis=0)
    print(f'{name:12s}' + ''.join(f'{v:9.3f}' for v in med))

# ① 밴드가 모든 지역에서 반사율 0–1 (원값 0–10000이 섞이면 실패)
p99 = max(np.percentile(S[n]['X'][:, :20], 99) for n in TRAIN_REGIONS)
assert p99 < 2.0, f'밴드 스케일 이상: 99% 분위 {p99:.1f}'
# ② 피처 수
assert all(S[n]['X'].shape[1] == 38 for n in TRAIN_REGIONS)
# ③ shift: α=0이면 원본, α=1이면 평가 중앙값 = 학습 중앙값
Xa = np.concatenate([S[n]['X'][:20000] for n in TRAIN_REGIONS[1:]])
Xb = S[TRAIN_REGIONS[0]]['X']
ma, sa = stats(Xa); mb, sb = stats(Xb)
assert shift(Xb, mb, sb, ma, sa, 0.0) is Xb
d1 = np.abs(np.median(shift(Xb, mb, sb, ma, sa, 1.0), axis=0) - ma).max()
assert d1 < 1e-3, d1
del Xa, Xb
print(f'\n검산 통과: 밴드 99% 분위 {p99:.2f} (0–1 범위), 피처 38개, shift α=0 원본 / α=1 중앙값 차 {d1:.1e}')

## 실험 2 사전 기록 (23절 2항)

**결과를 보기 전에 확정한다.** 아래 [6]은 인수인계 13·14·23절 값으로 미리 채워 두었다. 확인할 것:

- 합산 오차 기준 2.5% 채택 여부 (AI 제안값, 사용자 확정 필요)
- **잠금 시험 지역 선정 절차** — 22-3절 "실험 2 결과 보기 전에 정할 것". 비워 두면 저장되지 않는다
- 학습 샘플 수·평가 범위·시드

한 번 저장된 `exp2_prereg.json`은 이 셀이 덮어쓰지 않는다. 바꾸려면 파일을 직접 지우고, 바꾼 이유를 기록할 것.

In [ ]:
# [6] 사전 기록 — 확인·수정 후 CONFIRMED = True 로 바꾸고 실행
CONFIRMED = False

PREREG = {
    'date': time.strftime('%Y-%m-%d %H:%M'),
    'threshold_aggregate_pct': 2.5,
    'threshold_basis': '연간 임목축적 증가율 2.5% (2020 산림기본통계). 1년치 변화보다 큰 편향이면 증감 판단 불가 (13절 ①)',
    'prediction_sign': '북쪽 지역 +, 남쪽 지역 − (위도와 부호 오차의 순위상관 양수)',
    'prediction_aggregate': '면적가중 합산 |오차| ≤ 지역별 |오차| 평균의 절반',
    'judgement': '두 예측 모두 맞으면 초점은 시군 단위. 합산 오차도 기준 초과면 전국 단위 문제',
    'failure': '부호 패턴 없음, 또는 평균밀도 기준선이 분류 모델보다 나음 → 먼저 보고 (규칙 6)',
    'baseline_density': '학습 7지역 클래스 비율 × 확정계수 × 평가 지역 산림 픽셀 수',
    'aggregate': '지역 표본 탄소밀도 × 지역 유효 산림 픽셀 수(n_total)로 가중 합산',
    'seeds': SEEDS,
    'train_samples': f'학습 지역당 {N_TRAIN_PER_REGION:,} (7지역 합 {7 * N_TRAIN_PER_REGION:,}), 지역 동일 가중',
    'eval_scope': f'평가 지역 2024+ 숲 & 영상 유효 & 결측 {EDGE_PX}px 밖, 균등 표본 최대 {N_MAX:,}',
    'rf': {k: v for k, v in RF_PARAMS.items() if k != 'n_jobs'},
    'carbon_coeff': C,
    'lock_test_procedure': '미정 — 실행 전에 적을 것 (도별 학습 미사용 시군 1곳, 무작위 또는 사전 기준)',
}

pr_path = f'{OUT}/exp2_prereg.json'
if os.path.exists(pr_path):
    print('저장된 사전 기록 (이 셀은 덮어쓰지 않음):\n')
    print(json.dumps(json.load(open(pr_path, encoding='utf-8')), ensure_ascii=False, indent=1))
elif CONFIRMED:
    assert '미정' not in PREREG['lock_test_procedure'], '잠금 시험 선정 절차를 적어야 저장됩니다'
    json.dump(PREREG, open(pr_path, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print('사전 기록 저장:', pr_path)
else:
    print('사전 기록 미확정 — 내용을 확인하고 CONFIRMED = True 로 바꿔 이 셀을 다시 실행하세요.')

In [ ]:
# [7] 실험 2 — LORO 8지역 × 5시드 = 40폴드
#   폴드마다 파일 저장 → 끊기면 다시 실행하면 남은 폴드만 진행
#   같은 RF로 α 스윕 예측도 저장 (실험 3용, 이 노트북에서는 분석하지 않음)
assert os.path.exists(f'{OUT}/exp2_prereg.json'), '사전 기록 [6]을 먼저 확정하세요 (정상 정지)'

folds = [(s, r) for s in SEEDS for r in TRAIN_REGIONS]
todo  = [(s, r) for s, r in folds if not os.path.exists(f'{FOLD_DIR}/s{s}_{r}.npz')]
print(f'전체 {len(folds)}폴드 중 완료 {len(folds) - len(todo)}, 남은 {len(todo)}\n')

STATS_B = {r: stats(S[r]['X']) for r in TRAIN_REGIONS}
CVEC = np.array([C[1], C[2], C[3]])
times = []
for k, (seed, test) in enumerate(todo, 1):
    t0  = time.time()
    rng = np.random.default_rng(seed)
    Xtr, ytr = [], []
    for r in TRAIN_REGIONS:
        if r == test: continue
        n = len(S[r]['y'])
        i = rng.choice(n, size=min(N_TRAIN_PER_REGION, n), replace=False)
        Xtr.append(S[r]['X'][i]); ytr.append(S[r]['y'][i])
    Xtr, ytr = np.concatenate(Xtr), np.concatenate(ytr)
    ma, sa = stats(Xtr)
    prior  = np.array([(ytr == v).mean() for v in (1, 2, 3)])

    clf = RandomForestClassifier(random_state=seed, **RF_PARAMS)
    clf.fit(Xtr, ytr)
    del Xtr, ytr; gc.collect()

    Xb, yb = S[test]['X'], S[test]['y']
    mb, sb = STATS_B[test]
    t_c = carbon_sum(yb)
    carb, f1, p_c = [], [], []
    for a in ALPHAS:
        p = clf.predict(shift(Xb, mb, sb, ma, sa, a))
        p_c.append(carbon_sum(p))
        carb.append((p_c[-1] - t_c) * 100 / t_c)
        f1.append(f1_score(yb, p, average='macro', labels=[1, 2, 3]))
    base_err = (len(yb) * float(prior @ CVEC) - t_c) * 100 / t_c

    np.savez(f'{FOLD_DIR}/s{seed}_{test}.npz',
             alphas=np.array(ALPHAS), carb=np.array(carb), f1=np.array(f1), p_c=np.array(p_c),
             t_c=t_c, n_test=len(yb), base_err=base_err, prior=prior,
             n_total=int(S[test]['n_total']))
    del clf; gc.collect()

    dt = time.time() - t0; times.append(dt)
    eta = np.mean(times) * (len(todo) - k) / 60
    print(f'[{k:2d}/{len(todo)}] s{seed} {test:12s} 탄소오차(α=0) {carb[0]:+6.2f}%  F1 {f1[0]:.3f}  '
          f'기준선 {base_err:+6.2f}%   {dt:4.0f}초  남은 약 {eta:.0f}분')

print('\n모든 폴드 완료')

In [ ]:
# [8] 폴드 결과 모으기 → exp2_results.npz  (shape: 시드 × 지역 × α)
ns, nr, na = len(SEEDS), len(TRAIN_REGIONS), len(ALPHAS)
CARB = np.full((ns, nr, na), np.nan); F1 = np.full_like(CARB, np.nan); PC = np.full_like(CARB, np.nan)
TC   = np.full((ns, nr), np.nan);     NT = np.full_like(TC, np.nan);   BASE = np.full_like(TC, np.nan)
NTOT = np.array([int(S[r]['n_total']) for r in TRAIN_REGIONS], float)

for i, s in enumerate(SEEDS):
    for j, r in enumerate(TRAIN_REGIONS):
        f = f'{FOLD_DIR}/s{s}_{r}.npz'
        if not os.path.exists(f): continue
        z = np.load(f)
        CARB[i, j], F1[i, j], PC[i, j] = z['carb'], z['f1'], z['p_c']
        TC[i, j], NT[i, j], BASE[i, j] = z['t_c'], z['n_test'], z['base_err']

missing = int(np.isnan(TC).sum())
assert missing == 0, f'미완료 폴드 {missing}개 — [7]을 다시 실행'
np.savez(f'{OUT}/exp2_results.npz', carb=CARB, f1=F1, p_c=PC, t_c=TC, n_test=NT, base_err=BASE,
         n_total=NTOT, seeds=np.array(SEEDS), regions=np.array(TRAIN_REGIONS), alphas=np.array(ALPHAS))
print('저장:', f'{OUT}/exp2_results.npz', CARB.shape)

In [ ]:
# [9] 실험 2 결과 (α=0) — 지역별 / 면적가중 합산 / 평균밀도 기준선
j0 = ALPHAS.index(0.0)
e0 = CARB[:, :, j0]                                   # (시드, 지역) 부호 포함 %

tab = pd.DataFrame({
    'lat': [LAT[r] for r in TRAIN_REGIONS],
    'larch%': [LARCH[r] for r in TRAIN_REGIONS],
    'err_mean%': e0.mean(0),
    'err_sd': e0.std(0, ddof=1),
    'sign_same': [f'{int((np.sign(e0[:, j]) == np.sign(e0[:, j].mean())).sum())}/{ns}' for j in range(nr)],
    'F1': F1[:, :, j0].mean(0),
    'baseline_err%': BASE.mean(0),
}, index=TRAIN_REGIONS)
tab.to_csv(f'{OUT}/exp2_alpha0_by_region.csv')
print(tab.round(3).to_string())

dens_t = TC / NT                                       # 표본 탄소밀도 (실제)
dens_p = PC[:, :, j0] / NT                             # 표본 탄소밀도 (예측)
dens_b = (BASE / 100 + 1) * dens_t                     # 표본 탄소밀도 (기준선)
tot_t  = (dens_t * NTOT).sum(1)
agg      = ((dens_p * NTOT).sum(1) - tot_t) * 100 / tot_t
agg_base = ((dens_b * NTOT).sum(1) - tot_t) * 100 / tot_t
mean_abs = np.abs(e0).mean(1)

print('\n면적가중 합산 오차 (시드별):  ' + '  '.join(f'{x:+.2f}' for x in agg))
print(f'  RF 합산          {agg.mean():+.2f} ± {agg.std(ddof=1):.2f} %')
print(f'  지역별 |오차| 평균  {mean_abs.mean():.2f} ± {mean_abs.std(ddof=1):.2f} %')
print(f'  기준선 합산       {agg_base.mean():+.2f} %   기준선 지역별 |오차| 평균 {np.abs(BASE).mean():.2f} %')
pd.DataFrame({'seed': SEEDS, 'agg_rf%': agg, 'agg_baseline%': agg_base, 'mean_abs_region%': mean_abs}) \
  .to_csv(f'{OUT}/exp2_aggregate.csv', index=False)

In [ ]:
# [10] 사전 기록과 대조 — 수치만 나란히 보여준다. 판정은 사람이 한다
pr = json.load(open(f'{OUT}/exp2_prereg.json', encoding='utf-8'))
TH = pr['threshold_aggregate_pct']
latv   = [LAT[r] for r in TRAIN_REGIONS]
larchv = [LARCH[r] for r in TRAIN_REGIONS]
rho_lat   = [spearmanr(latv,   e0[i]).correlation for i in range(ns)]
rho_larch = [spearmanr(larchv, e0[i]).correlation for i in range(ns)]

print('예측 ① 북+/남− : 위도–오차 순위상관  ' + ' '.join(f'{x:+.2f}' for x in rho_lat)
      + f'   → 양수 {sum(x > 0 for x in rho_lat)}/{ns}')
print('   (참고) 낙엽송%–오차 순위상관   ' + ' '.join(f'{x:+.2f}' for x in rho_larch))
print(f'예측 ② 합산 ≤ 지역 평균의 절반 : {int((np.abs(agg) <= mean_abs / 2).sum())}/{ns} 시드')
print(f'기준 {TH}%  : 합산 |오차| 초과 {int((np.abs(agg) > TH).sum())}/{ns} 시드, '
      f'지역 |오차| 초과 {int((np.abs(e0) > TH).sum())}/{e0.size} 폴드')
print(f'실패 조건 : 기준선이 RF보다 나은 폴드 {int((np.abs(BASE) < np.abs(e0)).sum())}/{e0.size}, '
      f'합산에서 기준선이 나은 시드 {int((np.abs(agg_base) < np.abs(agg)).sum())}/{ns}')
print('\n13절 판정표: ① 합산 오차 > 기준이면 전국 단위 문제 / 작으면 α의 의미는 시군 단위로 좁혀짐')

In [ ]:
# [11] 그림 → exp2_alpha0.png  (Colab 기본 폰트에 한글이 없어 영문 표기)
lat = np.array(latv); o = np.argsort(lat)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

ax[0].axhline(0, c='k', lw=.8)
ax[0].errorbar(lat[o], e0.mean(0)[o], yerr=e0.std(0, ddof=1)[o], fmt='o-', capsize=4, label='RF (alpha=0)')
ax[0].plot(lat[o], BASE.mean(0)[o], 's--', c='gray', label='mean-density baseline')
for j in o:
    ax[0].annotate(TRAIN_REGIONS[j], (lat[j], e0.mean(0)[j]), fontsize=8, xytext=(3, 4), textcoords='offset points')
ax[0].set_xlabel('latitude'); ax[0].set_ylabel('signed carbon error (%)')
ax[0].set_title('LORO by region (mean ± sd over 5 seeds)'); ax[0].legend()

x = np.arange(ns)
ax[1].bar(x - .2, agg, width=.4, color='steelblue', label='RF')
ax[1].bar(x + .2, agg_base, width=.4, color='gray', label='baseline')
ax[1].axhline(TH, c='r', ls=':'); ax[1].axhline(-TH, c='r', ls=':'); ax[1].axhline(0, c='k', lw=.8)
ax[1].set_xticks(x); ax[1].set_xticklabels([f's{s}' for s in SEEDS])
ax[1].set_ylabel('area-weighted aggregate error (%)'); ax[1].set_title(f'Aggregate of 8 regions (dotted = ±{TH}%)')
ax[1].legend()

plt.tight_layout(); plt.savefig(f'{OUT}/exp2_alpha0.png', dpi=150); plt.show()

## 산출물 (`forest/outputs/`)

| 파일 | 내용 |
|---|---|
| `{지역}_s05.npz` ×8 | 22-5 마스크 반영 캐시 (지역당 60만, 약 70MB) |
| `exp2_region_meta.csv` | 지역별 유효 픽셀·클래스 비율·결측경계 제외 수 |
| `exp2_prereg.json` | 사전 기록 |
| `exp2_folds/s{시드}_{지역}.npz` ×40 | 폴드별 원자료 (α 스윕 포함) |
| `exp2_results.npz` | 전체 (시드 × 지역 × α) |
| `exp2_alpha0_by_region.csv`, `exp2_aggregate.csv`, `exp2_alpha0.png` | 실험 2 결과 |

## 다음

- 결과로 13절 판정(전국/시군) → 주장 범위 결정
- **실험 3**은 `exp2_results.npz`의 α 스윕을 읽어서 진행 (RF 재학습 없음). 간이 방식: 지역 r의 α = 나머지 폴드 α* 평균
- 06/07은 이전 05의 무효 결과를 전제로 짜여 있어 다시 작성해야 함